# Bundle 독립 검증

## 분석 목적
BE Producer의 schema, digest, cardinality, provenance와 집계를 DA에서 재검증한다.

## 가설
모든 실행이 고유 Case×Model 조합이고 섹션 간 연결 및 입력 digest가 일치한다.

## 사용할 변수
manifest, execution_config, case_results, runtime_metrics, failure_summary, trace_index

## 통계기법 선택 이유
통계 검정 대신 계약의 결정적 불변조건을 검사한다. 원본 바이트는 별도 보존한다.

## 해석 기준
모든 검사 PASS만 분석 허용. 실패하면 예외로 중단하며 값 보정은 하지 않는다. 전체 Case 누락은 외부 등록 catalog 없이 판별할 수 없고 digest는 전자서명이 아니다.

기본 입력은 **합성 Consumer 시험 fixture**이다. 실제 Bundle은 `ADP_AI_BUNDLE_SOURCE`로 지정한다. Case 독립성/대칭성은 자동 추정하지 않는다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "02_ai/src").is_dir())
sys.path.insert(0, str(ROOT / "02_ai/src"))
from adp_da.bundle_analysis import analyze_bundle  # noqa: E402
from adp_da.bundle_dataset import execution_dataframe  # noqa: E402
from adp_da.bundle_loader import load_bundle  # noqa: E402

source = os.environ.get("ADP_AI_BUNDLE_SOURCE")
synthetic = source is None or os.environ.get("ADP_AI_SYNTHETIC") == "1"
if source is None:
    source = str(ROOT / "02_ai/tests/fixtures/evaluation_bundle.synthetic.json")
bundle, metadata = load_bundle(
    source, ROOT / "02_ai/data/interim/ai_evaluation/raw",
    evaluation_run_id=os.environ.get("ADP_AI_EVALUATION_RUN_ID"),
    token=os.environ.get("ADP_BE_TOKEN"),
    remote_bearer_enabled=os.environ.get("ADP_BE_REMOTE_BEARER_AUTH_ENABLED") == "YES",
    local_admin_user_id=os.environ.get("ADP_BE_LOCAL_ADMIN_USER_ID"),
    local_admin_roles=os.environ.get("ADP_BE_LOCAL_ADMIN_ROLES"),
)
frame = execution_dataframe(bundle)
artifacts = analyze_bundle(
    frame, synthetic=synthetic,
    independent_cases=os.environ.get("ADP_AI_INDEPENDENT_CASES") == "1",
    symmetric_differences=os.environ.get("ADP_AI_SYMMETRIC_DIFFERENCES") == "1",
)
print("SYNTHETIC FIXTURE — SOFTWARE VALIDATION ONLY" if synthetic else "USER-SUPPLIED BUNDLE")
display({k: bundle["manifest"][k] for k in ("bundle_id", "content_digest", "execution_count")})


In [ ]:
display(metadata["validation"])
display(frame[["execution_id", "eval_case_id", "model_profile_id", "expected_input_digest",
               "actual_input_digest", "input_tokens", "output_tokens", "total_tokens"]])
